In [186]:
import numpy as np
import pandas as pd
import re
from sklearn.base import BaseEstimator, TransformerMixin


In [187]:
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

In [188]:
def sentence_metrics(text):
    sentences = [s.strip() for s in re.split(r'[.!?]+', text) if len(s.strip()) > 0]
    if not sentences:
        return 0, 0, 0
    
    words_per_sentence = [len(s.split()) for s in sentences]
    
    num_sentences = len(sentences)
    avg_words = np.mean(words_per_sentence)
    std_words = np.std(words_per_sentence)
    
    return num_sentences, avg_words, std_words

# humans often alternate long sentences with shorter ones... (I hope so)
sentence_metrics(train['TEXT'][0]), train['LABEL'][0]

((31, np.float64(16.870967741935484), np.float64(7.079010025027744)),
 np.int64(0))

In [189]:
import re
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin

class featureExtractor(BaseEstimator, TransformerMixin):
     def __init__(self):
          pass

     # humans often alternate long sentences with shorter ones... (I hope so)
     @staticmethod
     def __sentence_metrics(text):
          sentences = [s.strip() for s in re.split(r'[.!?]+', text) if len(s.strip()) > 0]
          if not sentences:
               return 0, 0, 0
          
          words_per_sentence = [len(s.split()) for s in sentences]
          
          num_sentences = len(sentences)
          avg_words = np.mean(words_per_sentence)
          std_words = np.std(words_per_sentence)
          return num_sentences, avg_words, std_words

     def fit(self, X, y=None):
          return self
     
     def transform(self, X):
          df = X.copy()
          
          df['TEXT_str'] = df['TEXT'].astype(str) 
          df['len'] = df['TEXT_str'].apply(len)
          df['words_list'] = df['TEXT_str'].str.split()
          
          df['num_words'] = df['words_list'].apply(lambda x: len(x) if isinstance(x, list) else 0)
          
          # avoiding to explode something in case of missing text
          df['len_safe'] = df['len'].replace(0, 1)
          df['words_safe'] = df['num_words'].replace(0, 1)

          # Punctuation...
          punctuation_chars = {
               'periods': '.', 'commas': ',', 'dashes': '-', 
               'question': '?', 'exclamation': '!', 'semicolon': ';', 'colon': ':',
               'spaces': ' ', 'newlines': '\n', 'asterisks': '*'
          }

          for name, char in punctuation_chars.items():
               df[name] = df['TEXT_str'].apply(lambda x: x.count(char))
               df[f'{name}_per_len'] = df[name] / df['len_safe']

          # mixed case like parenthesis and quotes
          df['parenthesis'] = df['TEXT_str'].apply(lambda x: x.count('(') + x.count(')'))
          df['parenthesis_per_len'] = df['parenthesis'] / df['len_safe']

          df['quotes'] = df['TEXT_str'].apply(lambda x: x.count('"') + x.count("'") + x.count('`'))
          df['quotes_per_len'] = df['quotes'] / df['len_safe']

          # uppercase and digits
          df['uppercase'] = df['TEXT_str'].apply(lambda x: len(re.findall(r'[A-Z]', x)))
          df['uppercase_per_len'] = df['uppercase'] / df['len_safe']

          df['digits'] = df['TEXT_str'].apply(lambda x: len(re.findall(r'\d', x)))
          df['digits_per_len'] = df['digits'] / df['len_safe']

          # unique words
          df['unique_words'] = df['words_list'].apply(lambda x: len(set(x)) if isinstance(x, list) else 0)
          df['unique_words_per_words'] = df['unique_words'] / df['words_safe']

          df['avg_word_length'] = df['words_list'].apply(
               lambda x: np.mean([len(w) for w in x]) if isinstance(x, list) and len(x) > 0 else 0
          )

          # enumerations
          df['enumeration_num'] = df['TEXT_str'].apply(lambda x: len(re.findall(r'\d+\.', x)))
          df['enumeration_num_per_len'] = df['enumeration_num'] / df['len_safe']

          metrics = df['TEXT_str'].apply(self.__sentence_metrics)
          df['num_sentences'] = [m[0] for m in metrics]
          df['sentences_per_len'] = df['num_sentences'] / df['len_safe']
          df['avg_words_per_sentence'] = [m[1] for m in metrics]
          df['std_sentence_length'] = [m[2] for m in metrics]

          # Cleaning
          df = df.drop(columns=['TEXT_str', 'words_list', 'len_safe', 'words_safe'])
          
          return df
     
     def fit_transform(self, X, y = None, **fit_params):
          return super().fit_transform(X, y, **fit_params)

In [190]:
from sklearn.model_selection import train_test_split
df_train, df_test = train_test_split(train, test_size=0.2, stratify=train['LABEL'], random_state=42)
X_train, y_train = df_train['TEXT'], df_train['LABEL']
X_test, y_test = df_test['TEXT'], df_test['LABEL']

In [191]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import chi2

class Chi2TextFeatureSelector(BaseEstimator, TransformerMixin):
     """
     This class applies TF-IDF (Term Frequency-Inverse Document Frequency) to text data
     and selects the most important features (words/n-grams) for each class using the 
     Chi-Square (Chi2) statistical test. It uses a One-vs-Rest strategy. (It was also applied in the DSMLL course by me)
     """
     
     def __init__(self, 
                    text_col: str = 'TEXT', 
                    k_per_label: int = 50, 
                    min_df: int = 3,  
                    ngram_range: tuple = (1, 2)):
          
          self.text_col = text_col
          # k_per_label: How many top features to select for each unique class.
          self.k_per_label = k_per_label
          # min_df: Minimum Document Frequency. Ignores words that appear in fewer than 'min_df' documents.
          self.min_df = min_df
          # ngram_range: (1, 2) means we extract single words (unigrams) and two-word phrases (bigrams).
          self.ngram_range = ngram_range
          
          # Attributes that will be learned during the fit()
          self.tfidf_vectorizer_ = None
          self.selected_indices_ = None 
          self.feature_names_ = None
          
     def fit(self, X: pd.DataFrame, y) -> 'Chi2TextFeatureSelector':
          """
          Learns the vocabulary from the text and selects the best features based on the Chi2 test.
          """
          # I absolutely need the labels to compute the Chi-Square statistical test!
          if y is None:
               raise ValueError("Target variable 'y' is required to compute Chi2.")
          
          y_arr = y.values if isinstance(y, pd.Series) else np.array(y)
          
          text_data = X[self.text_col].fillna('').astype(str)
          
          print(f"Fitting TF-IDF (min_df={self.min_df}, ngrams={self.ngram_range})...")
          
          self.tfidf_vectorizer_ = TfidfVectorizer(
               input='content', encoding='utf-8', lowercase=True,
               stop_words=None, # I want to keep stop words because they might be important for classification (e.g., "not", "but", "and") 
               min_df=self.min_df, 
               ngram_range=self.ngram_range
          )
          
         
          # Transform the text into a sparse matrix of TF-IDF scores
          X_tfidf = self.tfidf_vectorizer_.fit_transform(text_data)
          print(f"Selecting top {self.k_per_label} features per label via Chi-Square test...")
          unique_classes = np.unique(y_arr)
          
          feature_to_labels = {}

          # Computing Chi-Square for each class using the "One-vs-Rest" approach
          for label in unique_classes:
               # 1 if it's the current class, 0 otherwise
               y_binary = (y_arr == label).astype(int)
               
               # Compute the Chi2 scores between all TF-IDF features and the binary target
               # The chi2() function returns two arrays: scores and p-values. I only care about the scores.
               chi2_scores, _ = chi2(X_tfidf, y_binary)
               
               # Get the total number of available features in the TF-IDF vocabulary
               n_features = X_tfidf.shape[1]
               
               # Ensure we don't try to select more features than actually exist
               k_safe = min(self.k_per_label, n_features)

               if k_safe > 0:
                    top_k_indices = np.argsort(chi2_scores)[-k_safe:]
                    for idx in top_k_indices:
                         if idx not in feature_to_labels:
                              feature_to_labels[idx] = set()
                              # Record that this specific word index is a strong predictor for the current 'label'
                         feature_to_labels[idx].add(label)

          # Finalizing the selected indices and create descriptive column names
          # Extract all unique indices selected across all classes and sort them
          self.selected_indices_ = sorted(list(feature_to_labels.keys()))
          
          if not self.selected_indices_:
               print("Warning: No features were selected.")
               self.feature_names_ = []
               return self
               
          # Get the actual string words/n-grams from the fitted TF-IDF vocabulary
          raw_feature_names = self.tfidf_vectorizer_.get_feature_names_out()
          self.feature_names_ = []
          
          for idx in self.selected_indices_:
               # Extract the actual word corresponding to the numerical index
               word = raw_feature_names[idx]
               
               # Create a string suffix of the labels that this word helps predict (e.g., "0_3")
               labels_suffix = "_".join(sorted([str(lbl) for lbl in feature_to_labels[idx]]))
               
               # Construct the final descriptive feature name (e.g., "tfidf_apple_L0_3")
               self.feature_names_.append(f"tfidf_{word}_L{labels_suffix}")
               
          print(f"Total unique TF-IDF features selected: {len(self.selected_indices_)}")
          return self
     
     def transform(self, X: pd.DataFrame) -> pd.DataFrame:
          """
          Applies the learned TF-IDF transformation and filters the matrix to keep 
          only the features selected by the Chi2 test during the fit() phase.
          """
          # Check if the model has been fitted properly
          if self.tfidf_vectorizer_ is None:
               raise RuntimeError("The Transformer is not fitted yet. Call fit() before transform().")

          # Create a copy to avoid altering the original dataframe in memory
          df = X.copy()
          text_data = df[self.text_col].fillna('').astype(str)
          
          # Initialize a list of dataframes to concatenate at the end.
          # We start by removing the original raw text column so it doesn't get passed to the ML model.
          dfs_to_concat = [df.drop(columns=[self.text_col], errors='ignore')]
          
          if len(self.selected_indices_) > 0:
               # Transform the new text data using the ENTIRE vocabulary learned during fit()
               X_tfidf_full = self.tfidf_vectorizer_.transform(text_data)
               
               # Feature Selection (Filtering)
               # Slice the sparse matrix: keep ONLY the columns (indices) selected by the Chi2 test
               X_tfidf_sel = X_tfidf_full[:, self.selected_indices_]
               
               # Convert the sparse matrix into a dense Pandas DataFrame
               df_tfidf = pd.DataFrame(
                    X_tfidf_sel.toarray(),         # .toarray() converts the sparse matrix to a dense NumPy array
                    columns=self.feature_names_,   # Apply the descriptive column names we generated in fit()
                    index=df.index                 # Keep the original index to ensure alignment with other features
               )
               
               # Add the new TF-IDF numerical dataframe to our concatenation list
               dfs_to_concat.append(df_tfidf)
               
          # Concatenate horizontally (axis=1) to combine any existing features with the new text features
          final_df = pd.concat(dfs_to_concat, axis=1)
          return final_df

In [192]:
X_train = pd.DataFrame(X_train)

In [193]:
X_train_aug = featureExtractor().fit_transform(X_train, y_train)

In [194]:
impCHI = Chi2TextFeatureSelector().fit(X_train, y_train)
X_train_pp = impCHI.transform(X_train_aug)
X_train_pp

Fitting TF-IDF (min_df=3, ngrams=(1, 2))...
Selecting top 50 features per label via Chi-Square test...
Total unique TF-IDF features selected: 282


,len,num_words,periods,periods_per_len,commas,commas_per_len,dashes,dashes_per_len,question,question_per_len,...,tfidf_vital_L4,tfidf_vital role_L4,tfidf_vote_L0,tfidf_was_L0,tfidf_water_L3,tfidf_watt_L2,tfidf_we can_L4,tfidf_would_L0,tfidf_yeast_L2,tfidf_you_L0
942,1189,135,8,0.006728,26,0.021867,1,0.000841,0,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000
1227,214,34,1,0.004673,3,0.014019,0,0.000000,0,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000
2108,5226,781,37,0.007080,48,0.009185,10,0.001914,0,0.000000,...,0.035615,0.040381,0.000000,0.000000,0.016577,0.0,0.042832,0.000000,0.0,0.000000
345,664,84,4,0.006024,6,0.009036,3,0.004518,0,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000
2072,1031,178,11,0.010669,9,0.008729,0,0.000000,5,0.004850,...,0.000000,0.000000,0.000000,0.101740,0.000000,0.0,0.000000,0.000000,0.0,0.027041
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
245,562,86,6,0.010676,4,0.007117,2,0.003559,0,0.000000,...,0.074773,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.076429
2239,3947,676,27,0.006841,32,0.008107,0,0.000000,2,0.000507,...,0.000000,0.000000,0.222089,0.018428,0.000000,0.0,0.000000,0.041656,0.0,0.019591
1789,3874,731,33,0.008518,26,0.006711,0,0.000000,4,0.001033,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.016607,0.078778,0.0,0.074101
126,3853,667,43,0.011160,40,0.010382,0,0.000000,1,0.000260,...,0.000000,0.000000,0.000000,0.030805,0.000000,0.0,0.000000,0.023212,0.0,0.043667


In [195]:
cols_name = X_train_pp.columns.tolist()

In [196]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pipeline = Pipeline(
     [
          
          ('feature_ext', featureExtractor()),
          ('impChi', Chi2TextFeatureSelector(k_per_label=60, min_df=5, ngram_range=(1, 2))),
          ('scaler', StandardScaler()) # probably not needed...
     ]
)

X_train_pp = pipeline.fit_transform(pd.DataFrame(X_train), y_train)
X_test_pp = pipeline.transform(pd.DataFrame(X_test))

Fitting TF-IDF (min_df=5, ngrams=(1, 2))...
Selecting top 60 features per label via Chi-Square test...
Total unique TF-IDF features selected: 334


In [197]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import f1_score
hgb = HistGradientBoostingClassifier(class_weight='balanced', scoring='f1_macro')
hgb.fit(X_train_pp, y_train)
y_pred = hgb.predict(X_test_pp)
f1_score(y_test, y_pred, average='macro')

0.9079110370387959

In [198]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC


models = {
     'svm': SVC(class_weight='balanced'),
     'rf': RandomForestClassifier(class_weight='balanced', n_estimators=100, random_state=42),
     'logistic': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
     'knn': KNeighborsClassifier(n_neighbors=5),
     'hgb': HistGradientBoostingClassifier(class_weight='balanced', random_state=42),
}

for name, model in models.items():
     print(f"Training {name}...")
     model.fit(X_train_pp, y_train)
     y_pred = model.predict(X_test_pp)
     score = f1_score(y_test, y_pred, average='macro')
     print(f"{name} F1 Macro: {score:.4f}\n")

Training svm...
svm F1 Macro: 0.8538

Training rf...
rf F1 Macro: 0.9334

Training logistic...
logistic F1 Macro: 0.8604

Training knn...
knn F1 Macro: 0.7242

Training hgb...
hgb F1 Macro: 0.9079



In [202]:
import optuna
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.ensemble import HistGradientBoostingClassifier

extractor = featureExtractor()
X_train_syntax = extractor.fit_transform(pd.DataFrame(X_train))

def objective(trial):
    k_per_label = trial.suggest_int('k_per_label', 20, 80, step=10)
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
    max_iter = trial.suggest_int('max_iter', 100, 300, step=50)
    max_depth = trial.suggest_categorical('max_depth', [None, 3, 5, 7, 9])
    l2_regularization = trial.suggest_float('l2_regularization', 0.0, 5.0)

    pipeline = Pipeline([
        ('impChi', Chi2TextFeatureSelector(
            text_col='TEXT',
            k_per_label=k_per_label, 
            min_df=5, 
            ngram_range=(1, 2)
        )),
        ('hgb', HistGradientBoostingClassifier(
            learning_rate=learning_rate,
            max_iter=max_iter,
            max_depth=max_depth,
            l2_regularization=l2_regularization,
            class_weight='balanced',
            random_state=42
        ))
    ])

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = cross_val_score(
        pipeline, 
        X_train_syntax, 
        y_train, 
        cv=cv, 
        scoring='f1_macro', 
        n_jobs=-1
    )
    
    return scores.mean()

# Disabilitiamo i log prolissi di Optuna e usiamo solo la barra di progresso
optuna.logging.set_verbosity(optuna.logging.WARNING)

study = optuna.create_study(direction='maximize')
print("Avvio ottimizzazione Optuna...")

# Imposta n_trials al numero di iterazioni che vuoi fare (es. 30 o 50)
study.optimize(objective, n_trials=30, show_progress_bar=True)

print(f"\nBest F1 Macro Score: {study.best_value:.4f}")
print("Best Parameters:")
for param, value in study.best_params.items():
    print(f" - {param}: {value}")

Avvio ottimizzazione Optuna...


  0%|          | 0/30 [00:00<?, ?it/s]

[W 2026-03-12 18:13:48,895] Trial 24 failed with parameters: {'k_per_label': 60, 'learning_rate': 0.031674258400816904, 'max_iter': 250, 'max_depth': 9, 'l2_regularization': 1.0527916796587715} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\gianb\AppData\Local\Programs\Python\Python313\Lib\site-packages\optuna\study\_optimize.py", line 205, in _run_trial
    value_or_values = func(trial)
  File "C:\Users\gianb\AppData\Local\Temp\ipykernel_23664\189737567.py", line 35, in objective
    scores = cross_val_score(
        pipeline,
    ...<4 lines>...
        n_jobs=-1
    )
  File "c:\Users\gianb\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\_param_validation.py", line 218, in wrapper
    return func(*args, **kwargs)
  File "c:\Users\gianb\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\model_selection\_validation.py", line 677, in cross_val_score
    cv_results = cross_validate(
        

KeyboardInterrupt: 

In [ ]:
import pandas as pd
import numpy as np
import optuna
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold
from lightgbm import LGBMClassifier

extractor = featureExtractor()
X_train_syntax = extractor.fit_transform(pd.DataFrame(X_train))
y_train_arr = y_train.values if hasattr(y_train, 'values') else np.array(y_train)

def objective(trial):
    k_per_label = trial.suggest_int('k_per_label', 15, 60, step=5)

    param = {
        'objective': 'multiclass',
        'num_class': 6,
        'class_weight': 'balanced',
        'metric': 'multi_logloss',
        'boosting_type': 'gbdt',
        'random_state': 42,
        'n_jobs': -1,
        'verbosity': -1,
        'n_estimators': trial.suggest_int('n_estimators', 400, 800, step=100),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.05, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 150, 300),
        'max_depth': trial.suggest_int('max_depth', 10, 18),
        'min_child_samples': trial.suggest_int('min_child_samples', 40, 100),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.01, 0.5, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1.0, 10.0, log=True),
        'min_split_gain': trial.suggest_float('min_split_gain', 0.01, 0.2),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 0.6),
        'subsample': trial.suggest_float('subsample', 0.7, 0.9),
        'subsample_freq': trial.suggest_int('subsample_freq', 1, 10),
    }

    pipeline = Pipeline([
        ('impChi', Chi2TextFeatureSelector(
            text_col='TEXT', 
            k_per_label=k_per_label, 
            min_df=5
        )),
        ('lgbm', LGBMClassifier(**param))
    ])

    # Cross-Validation
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = cross_val_score(
        pipeline, 
        X_train_syntax, 
        y_train_arr, 
        cv=cv, 
        scoring='f1_macro', 
        n_jobs=-1
    )
    
    return scores.mean()

optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction='maximize')

print("Starting bayesian optim...")
study.optimize(objective, n_trials=50, show_progress_bar=True)

print("\n" + "="*30)
print("Completed Hyperparameter Optimization")
print("="*30)
print(f"best F1 Macro (CV): {study.best_value:.4f}")
print("\nbest params:")
for key, value in study.best_params.items():
    print(f" - {key}: {value}")

print("\nFinal training...")
best_k = study.best_params.pop('k_per_label')

final_pipeline = Pipeline([
    ('impChi', Chi2TextFeatureSelector(text_col='TEXT', k_per_label=best_k, min_df=5)),
    ('lgbm', LGBMClassifier(**study.best_params, objective='multiclass', class_weight='balanced', random_state=42))
])

final_pipeline.fit(X_train_syntax, y_train_arr)

X_test_syntax = extractor.transform(pd.DataFrame(X_test))
y_pred = final_pipeline.predict(X_test_syntax)

from sklearn.metrics import f1_score, classification_report
final_f1 = f1_score(y_test, y_pred, average='macro')
print(f"\nF1 Macro on TEST SET: {final_f1:.4f}")
print("\nReport:")
print(classification_report(y_test, y_pred))

Estrazione feature sintattiche in corso...
Inizio Ottimizzazione Bayesiana con LightGBM...


  0%|          | 0/50 [00:00<?, ?it/s]


OTTIMIZZAZIONE COMPLETATA
Miglior F1 Macro (CV): 0.9249

Parametri migliori trovati:
 - k_per_label: 45
 - n_estimators: 400
 - learning_rate: 0.023498688458820725
 - num_leaves: 248
 - max_depth: 12
 - min_child_samples: 67
 - reg_alpha: 0.21539596093227129
 - reg_lambda: 1.2044291600666095
 - min_split_gain: 0.04398702675656692
 - colsample_bytree: 0.33712454303399336
 - subsample: 0.887193186722178
 - subsample_freq: 4

Addestramento modello finale sui dati di train...
Fitting TF-IDF (min_df=5, ngrams=(1, 2))...
Selecting top 45 features per label via Chi-Square test...
Total unique TF-IDF features selected: 252

F1 Macro finale su TEST SET: 0.9402

Report di classificazione:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       304
           1       0.76      0.81      0.79        16
           2       0.96      0.84      0.90        32
           3       1.00      1.00      1.00        16
           4       0.96      1.00      0